# Stage 2: Pothole & Core Segmentation Training (Dual MobileNetV3 + LTCA + Alpha Gate + Topological Loss)
### Component C2: Software Emulated Linear Transposed Cross Attention Framework for Noise Resilient Pavement Segmentation

**Project Title:** AI Framework for Sri Lankan Road Management (RoadLens)  
**Research Group:** J26-DS-309  
**Researcher:** Methma C A N (IT23245860) – B.Sc. (Hons) in Information Technology Specializing in Data Science, SLIIT  
**Supervisors:** Ms. Thamali Dassanayake & Ms. Hansi De Silva  

---
### Stage 2 Innovations:
1. **Software Thermal Sensor:** Generates pseudo-thermal moisture distribution via frozen cGAN.
2. **Dual-Stream MobileNetV3 Backbones:** Extracts spatial details and moisture features.
3. **Linear Transposed Cross-Attention (LTCA):** $O(N)$ channel cross-covariance attention.
4. **Dynamic Alpha Gate:** Blends contextual noise rejection against fine spatial boundaries.
5. **Skeletal Topological Loss:** Enforces centerline continuity via differentiable soft-skeletonization.
6. **4-Part Output Suite:** 2D Binary Mask, 1-Pixel Skeleton, Soft Confidence Map, and Red Visual Overlay.

## 1. Hardware Verification & GPU Configuration

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    torch.backends.cudnn.benchmark = True
    print("cuDNN Benchmark enabled.")
else:
    print("WARNING: CUDA not detected! Change runtime type to GPU.")

## 2. Dependencies Installation

In [ ]:
!pip install -q scikit-image albumentations opencv-python matplotlib tqdm torchvision

## 3. Google Drive Mount & High-Speed Local SSD Staging

In [ ]:
import os
import shutil
from pathlib import Path

try:
    from google.colab import drive
    drive_mount_point = "/content/drive"
    if not os.path.exists(drive_mount_point):
        drive.mount(drive_mount_point)
        print("Google Drive mounted.")
    else:
        print(f"Google Drive already mounted at {drive_mount_point}.")
    DRIVE_ROOT = Path("/content/drive/MyDrive/C2_Pavement_Research")
except ImportError:
    print("Running in local environment.")
    DRIVE_ROOT = Path("./C2_Pavement_Research")

WEIGHTS_DIR = DRIVE_ROOT / "weights"
CGAN_WEIGHTS = WEIGHTS_DIR / "cgan_frozen_sensor.pth"
if not CGAN_WEIGHTS.exists():
    CGAN_WEIGHTS = WEIGHTS_DIR / "cgan_generator_best.pth"

CORE_SEG_DRIVE = DRIVE_ROOT / "datasets" / "Core_Segmentation"
EXP_DIR = DRIVE_ROOT / "experiments" / "core_segmentation"
SUITE_DEMOS_DIR = EXP_DIR / "suite_demos"

for d in [WEIGHTS_DIR, EXP_DIR, SUITE_DEMOS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

LOCAL_DATASET = Path("/content/local_core_dataset")
if CORE_SEG_DRIVE.exists():
    if not LOCAL_DATASET.exists():
        print("Staging Core_Segmentation to local high-speed SSD...")
        shutil.copytree(str(CORE_SEG_DRIVE), str(LOCAL_DATASET))
        print("Local SSD staging complete!")
    DATASET_ROOT = LOCAL_DATASET
else:
    DATASET_ROOT = LOCAL_DATASET if LOCAL_DATASET.exists() else CORE_SEG_DRIVE

TRAIN_DIR = DATASET_ROOT / "train"
VAL_DIR = DATASET_ROOT / "val"
TEST_DIR = DATASET_ROOT / "test"

print(f"Trained cGAN Sensor Weights: {CGAN_WEIGHTS} (Exists: {CGAN_WEIGHTS.exists()})")
print(f"Core Segmentation Train Dir: {TRAIN_DIR} (Exists: {TRAIN_DIR.exists()})")
print(f"Core Segmentation Val Dir:   {VAL_DIR} (Exists: {VAL_DIR.exists()})")
print(f"Core Segmentation Test Dir:  {TEST_DIR} (Exists: {TEST_DIR.exists()})")

## 4. Loading & Verifying the Frozen cGAN Software Sensor

In [ ]:
import torch
import torch.nn as nn

class UNetDownBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, normalize: bool = True):
        super().__init__()
        layers = [nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=not normalize)]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_channels))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)

class UNetUpBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(out_channels),
            nn.ReLU(inplace=True)
        ]
        if dropout > 0.0:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        return torch.cat([self.block(x), skip], dim=1)

class UNetGenerator256(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        self.down1 = UNetDownBlock(in_channels, 64, normalize=False)
        self.down2 = UNetDownBlock(64, 128)
        self.down3 = UNetDownBlock(128, 256)
        self.down4 = UNetDownBlock(256, 512)
        self.down5 = UNetDownBlock(512, 512)
        self.down6 = UNetDownBlock(512, 512)
        self.down7 = UNetDownBlock(512, 512)
        self.down8 = UNetDownBlock(512, 512, normalize=False)
        self.up1 = UNetUpBlock(512, 512, dropout=0.5)
        self.up2 = UNetUpBlock(1024, 512, dropout=0.5)
        self.up3 = UNetUpBlock(1024, 512, dropout=0.5)
        self.up4 = UNetUpBlock(1024, 512)
        self.up5 = UNetUpBlock(1024, 256)
        self.up6 = UNetUpBlock(512, 128)
        self.up7 = UNetUpBlock(256, 64)
        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, out_channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        d1 = self.down1(x); d2 = self.down2(d1); d3 = self.down3(d2); d4 = self.down4(d3)
        d5 = self.down5(d4); d6 = self.down6(d5); d7 = self.down7(d6); d8 = self.down8(d7)
        u1 = self.up1(d8, d7); u2 = self.up2(u1, d6); u3 = self.up3(u2, d5); u4 = self.up4(u3, d4)
        u5 = self.up5(u4, d3); u6 = self.up6(u5, d2); u7 = self.up7(u6, d1)
        return self.final(u7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cgan_sensor = UNetGenerator256(in_channels=3, out_channels=1).to(device)

if CGAN_WEIGHTS.exists():
    state = torch.load(CGAN_WEIGHTS, map_location=device, weights_only=False)
    if "generator_state_dict" in state:
        state = state["generator_state_dict"]
    cgan_sensor.load_state_dict(state)
    print(f"Successfully loaded trained cGAN weights from {CGAN_WEIGHTS}")
else:
    print("Warning: cGAN weights file not found. Initialized weights will be used.")

for p in cgan_sensor.parameters():
    p.requires_grad = False
cgan_sensor.eval()
print("cGAN Software Sensor is FROZEN (requires_grad = False). Ready for Stage 2.")

## 5. Core Segmentation Dataset & Ground-Truth Skeletonization

In [ ]:
import cv2
import re
import numpy as np
import random
from pathlib import Path
from typing import List, Tuple
from torch.utils.data import Dataset, DataLoader

def clean_stem(stem: str) -> str:
    stem = stem.lower()
    for tag in ["_mask", "-mask", "_gt", "-gt", "_seg", "-seg", "_label", "-label", "mask", "label"]:
        stem = re.sub(rf"{re.escape(tag)}$", "", stem)
    return stem

def discover_segmentation_pairs(split_dir: Path) -> List[Tuple[Path, Path]]:
    split_dir = Path(split_dir)
    if not split_dir.exists():
        return []
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    img_candidates = {"images", "image", "imgs", "img", "rgb", "data", "inputs"}
    mask_candidates = {"masks", "mask", "labels", "label", "gt", "groundtruth", "seg", "annotations"}

    # Subdirectory discovery (supports 'train_images', 'val_masks', etc.)
    img_dir, mask_dir = None, None
    for sub in split_dir.iterdir():
        if sub.is_dir():
            nl = sub.name.lower()
            if any(k in nl for k in img_candidates) and img_dir is None:
                img_dir = sub
            elif any(k in nl for k in mask_candidates) and mask_dir is None:
                mask_dir = sub

    if img_dir and mask_dir:
        imgs = {p.stem: p for p in img_dir.iterdir() if p.suffix.lower() in valid_exts}
        msks = {p.stem: p for p in mask_dir.iterdir() if p.suffix.lower() in valid_exts}
        common = sorted(list(set(imgs.keys()) & set(msks.keys())))
        if len(common) > 0:
            return [(imgs[k], msks[k]) for k in common]
        img_norm = {clean_stem(k): v for k, v in imgs.items()}
        msk_norm = {clean_stem(k): v for k, v in msks.items()}
        common_norm = sorted(list(set(img_norm.keys()) & set(msk_norm.keys())))
        if len(common_norm) > 0:
            return [(img_norm[k], msk_norm[k]) for k in common_norm]

    # Recursive match fallback
    all_files = [p for p in split_dir.rglob("*") if p.suffix.lower() in valid_exts]
    imgs, msks = {}, {}
    for p in all_files:
        stem_norm = clean_stem(p.stem)
        nl = p.stem.lower()
        if any(tag in nl for tag in ["mask", "gt", "label", "seg"]) or (p.parent.name.lower() in mask_candidates):
            msks[stem_norm] = p
        else:
            imgs[stem_norm] = p
    common = sorted(list(set(imgs.keys()) & set(msks.keys())))
    if len(common) > 0:
        return [(imgs[k], msks[k]) for k in common]

    return []

def morphological_thinning(mask: np.ndarray) -> np.ndarray:
    binary = ((mask > 127) * 255).astype(np.uint8)
    try:
        skeleton = cv2.ximgproc.thinning(binary, thinningType=cv2.ximgproc.THINNING_ZHANGSUEN)
    except (AttributeError, cv2.error):
        kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
        skeleton = np.zeros_like(binary)
        temp = binary.copy()
        while True:
            eroded = cv2.erode(temp, kernel)
            opened = cv2.morphologyEx(eroded, cv2.MORPH_OPEN, kernel)
            subset = cv2.subtract(eroded, opened)
            skeleton = cv2.bitwise_or(skeleton, subset)
            temp = eroded.copy()
            if cv2.countNonZero(temp) == 0:
                break
    return (skeleton > 0).astype(np.float32)

class CoreSegmentationDataset(Dataset):
    def __init__(self, split_dir: Path, img_size: int = 256, is_train: bool = True):
        self.split_dir = Path(split_dir)
        self.img_size = img_size
        self.is_train = is_train
        self.pairs = discover_segmentation_pairs(self.split_dir)

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        img_path, mask_path = self.pairs[idx]

        rgb = cv2.imread(str(img_path))
        if rgb is None:
            raise ValueError(f"Failed to load image: {img_path}")
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            mask = cv2.imread(str(mask_path))
            mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

        rgb = cv2.resize(rgb, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        if self.is_train:
            if random.random() > 0.5:
                rgb = np.fliplr(rgb); mask = np.fliplr(mask)
            if random.random() > 0.5:
                rgb = np.flipud(rgb); mask = np.flipud(mask)
            if random.random() > 0.5:
                k = random.choice([1, 2, 3])
                rgb = np.rot90(rgb, k); mask = np.rot90(mask, k)

        skel = morphological_thinning(mask)

        rgb_t = torch.from_numpy(rgb.copy()).permute(2, 0, 1).float().div_(127.5).sub_(1.0)
        mask_t = torch.from_numpy((mask > 127).astype(np.float32)).unsqueeze(0)
        skel_t = torch.from_numpy(skel).unsqueeze(0)
        return rgb_t, mask_t, skel_t

train_dataset = CoreSegmentationDataset(TRAIN_DIR, img_size=256, is_train=True)
val_dataset = CoreSegmentationDataset(VAL_DIR, img_size=256, is_train=False)
test_dataset = CoreSegmentationDataset(TEST_DIR, img_size=256, is_train=False)

print(f"Discovered: Train={len(train_dataset)} pairs | Val={len(val_dataset)} pairs | Test={len(test_dataset)} pairs")

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## 6. Inspecting Training Samples & Software Thermal Synthesis

In [ ]:
import matplotlib.pyplot as plt

def denorm(t: torch.Tensor) -> np.ndarray:
    return np.clip((t.detach().cpu().numpy() + 1.0) / 2.0, 0.0, 1.0)

sample_rgb, sample_mask, sample_skel = train_dataset[0]
with torch.no_grad():
    sample_thm = cgan_sensor(sample_rgb.unsqueeze(0).to(device)).squeeze(0)

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(np.transpose(denorm(sample_rgb), (1, 2, 0)))
ax[0].set_title("Input RGB Optical Crop")
ax[0].axis("off")

ax[1].imshow(denorm(sample_thm).squeeze(), cmap="inferno")
ax[1].set_title("Synthesized Pseudo-Thermal")
ax[1].axis("off")

ax[2].imshow(sample_mask.squeeze().cpu().numpy(), cmap="gray")
ax[2].set_title("Ground Truth 2D Mask")
ax[2].axis("off")

ax[3].imshow(sample_skel.squeeze().cpu().numpy(), cmap="hot")
ax[3].set_title("1-Pixel Skeleton Centerline")
ax[3].axis("off")

plt.tight_layout()
plt.show()

## 7. Dual-Stream MobileNetV3 + LTCA + Alpha Gate Architecture

In [ ]:
import torch.nn.functional as F
import torchvision.models as models

class LinearTransposedCrossAttention(nn.Module):
    def __init__(self, dim: int = 96, num_heads: int = 4):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))

        self.q_proj = nn.Conv2d(dim, dim, kernel_size=1, bias=False)
        self.q_dwconv = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim, bias=False)
        self.k_proj = nn.Conv2d(dim, dim, kernel_size=1, bias=False)
        self.k_dwconv = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim, bias=False)
        self.v_proj = nn.Conv2d(dim, dim, kernel_size=1, bias=False)
        self.v_dwconv = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim, bias=False)
        self.out_proj = nn.Conv2d(dim, dim, kernel_size=1, bias=False)

        self.norm_rgb = nn.GroupNorm(1, dim)
        self.norm_thm = nn.GroupNorm(1, dim)
        self.norm_ffn = nn.GroupNorm(1, dim)
        self.ffn = nn.Sequential(
            nn.Conv2d(dim, dim * 2, kernel_size=1, bias=False),
            nn.GELU(),
            nn.Conv2d(dim * 2, dim, kernel_size=1, bias=False)
        )

    def forward(self, x_rgb: torch.Tensor, x_thm: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x_rgb.shape
        residual = x_rgb

        q = self.q_dwconv(self.q_proj(self.norm_rgb(x_rgb))).reshape(b, self.num_heads, self.head_dim, h * w)
        k = self.k_dwconv(self.k_proj(self.norm_thm(x_thm))).reshape(b, self.num_heads, self.head_dim, h * w)
        v = self.v_dwconv(self.v_proj(self.norm_thm(x_thm))).reshape(b, self.num_heads, self.head_dim, h * w)

        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)
        attn = F.softmax(torch.matmul(q, k.transpose(-2, -1)) * self.temperature, dim=-1)
        out = torch.matmul(attn, v).reshape(b, c, h, w)
        x = residual + self.out_proj(out)
        return x + self.ffn(self.norm_ffn(x))

class DynamicAlphaGate(nn.Module):
    def __init__(self, channels: int = 96):
        super().__init__()
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.channel_mlp = nn.Sequential(
            nn.Linear(channels * 2, channels // 2, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // 2, 1, bias=True),
            nn.Sigmoid()
        )
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(channels * 2, channels // 4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels // 4),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 4, 1, kernel_size=1, bias=True),
            nn.Sigmoid()
        )
        self.refine = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, f_spatial: torch.Tensor, f_context: torch.Tensor):
        b, c, h, w = f_spatial.shape
        combined = torch.cat([f_spatial, f_context], dim=1)
        global_alpha = self.channel_mlp(self.global_pool(combined).view(b, c * 2)).view(b, 1, 1, 1)
        local_alpha = self.spatial_conv(combined)
        alpha = 0.5 * global_alpha + 0.5 * local_alpha
        f_gated = (alpha * f_context) + ((1.0 - alpha) * f_spatial)
        return self.refine(f_gated), alpha

class DecoderBlock(nn.Module):
    def __init__(self, in_c: int, skip_c: int, out_c: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c + skip_c, out_c, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x: torch.Tensor, skip: torch.Tensor | None = None):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        if skip is not None:
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.relu(self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))))

class DualStreamLTCASegmenter(nn.Module):
    def __init__(self, cgan_sensor_module: nn.Module, feature_dim: int = 96):
        super().__init__()
        self.cgan_sensor = cgan_sensor_module

        m_rgb = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        m_thm = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)

        self.rgb_low = nn.Sequential(*m_rgb.features[:4])    # 1/4 scale, 24 channels
        self.thm_low = nn.Sequential(*m_thm.features[:4])
        self.rgb_high = nn.Sequential(*m_rgb.features[4:12]) # 1/16 scale, 96 channels
        self.thm_high = nn.Sequential(*m_thm.features[4:12])
        self.thm_stem = nn.Conv2d(1, 3, kernel_size=1, bias=False)

        self.ltca = LinearTransposedCrossAttention(dim=feature_dim, num_heads=4)
        self.spatial_proj = nn.Sequential(
            nn.Conv2d(24, feature_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(feature_dim),
            nn.ReLU(inplace=True)
        )
        self.alpha_gate = DynamicAlphaGate(channels=feature_dim)

        self.dec_high = DecoderBlock(feature_dim, 0, 48)
        self.dec_mid = DecoderBlock(48, feature_dim, 24)
        self.dec_final = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(24, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(16, 1, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, rgb: torch.Tensor):
        with torch.no_grad():
            pseudo_thm = self.cgan_sensor(rgb)

        rgb_norm = (rgb + 1.0) * 0.5
        thm_3ch = self.thm_stem((pseudo_thm + 1.0) * 0.5)

        f_rgb_low = self.rgb_low(rgb_norm)
        f_thm_low = self.thm_low(thm_3ch)

        f_rgb_high = self.rgb_high(f_rgb_low)
        f_thm_high = self.thm_high(f_thm_low)

        f_fused_high = self.ltca(f_rgb_high, f_thm_high)
        f_spatial = self.spatial_proj(f_rgb_low)
        f_context_up = F.interpolate(f_fused_high, size=f_spatial.shape[-2:], mode="bilinear", align_corners=False)

        f_gated_spatial, alpha = self.alpha_gate(f_spatial, f_context_up)

        d1 = self.dec_high(f_fused_high)
        d2 = self.dec_mid(d1, f_gated_spatial)
        prob_map = self.dec_final(d2)

        # Dimension guard: ensures (256, 256) alignment with input
        if prob_map.shape[-2:] != rgb.shape[-2:]:
            prob_map = F.interpolate(prob_map, size=rgb.shape[-2:], mode="bilinear", align_corners=False)

        return prob_map, alpha, pseudo_thm

segmenter = DualStreamLTCASegmenter(cgan_sensor_module=cgan_sensor, feature_dim=96).to(device)
print(f"Segmenter initialized on {device}.")

## 8. Skeletal Topological Loss (AMP Safe)

In [ ]:
def soft_erode(img: torch.Tensor) -> torch.Tensor:
    p1 = -F.max_pool2d(-img, kernel_size=(3, 1), stride=1, padding=(1, 0))
    p2 = -F.max_pool2d(-img, kernel_size=(1, 3), stride=1, padding=(0, 1))
    return torch.min(p1, p2)

def soft_dilate(img: torch.Tensor) -> torch.Tensor:
    p1 = F.max_pool2d(img, kernel_size=(3, 1), stride=1, padding=(1, 0))
    p2 = F.max_pool2d(img, kernel_size=(1, 3), stride=1, padding=(0, 1))
    return torch.max(p1, p2)

def soft_open(img: torch.Tensor) -> torch.Tensor:
    return soft_dilate(soft_erode(img))

def soft_skeletonize(x: torch.Tensor, iterations: int = 4) -> torch.Tensor:
    img = x
    skel = F.relu(img - soft_open(img))
    for _ in range(iterations):
        img = soft_erode(img)
        skel = skel + F.relu(img - soft_open(img))
    return torch.clamp(skel, 0.0, 1.0)

class SoftclDiceLoss(nn.Module):
    def __init__(self, iter_: int = 4, smooth: float = 1.0):
        super().__init__()
        self.iter = iter_
        self.smooth = smooth
    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor, y_skel: torch.Tensor):
        s_pred = soft_skeletonize(y_pred, iterations=self.iter)
        s_true = y_skel
        t_prec = (torch.sum(s_pred * y_true, dim=(1, 2, 3)) + self.smooth) / (torch.sum(s_pred, dim=(1, 2, 3)) + self.smooth)
        t_sens = (torch.sum(s_true * y_pred, dim=(1, 2, 3)) + self.smooth) / (torch.sum(s_true, dim=(1, 2, 3)) + self.smooth)
        cl_dice = (2.0 * t_prec * t_sens) / (t_prec + t_sens + 1e-7)
        return 1.0 - torch.mean(cl_dice)

class SoftDiceLoss(nn.Module):
    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor):
        inter = torch.sum(y_pred * y_true, dim=(1, 2, 3))
        card = torch.sum(y_pred + y_true, dim=(1, 2, 3))
        return 1.0 - torch.mean((2.0 * inter + 1.0) / (card + 1.0))

class SafeCompositeTopologicalSegmentationLoss(nn.Module):
    def __init__(self, a: float = 1.0, b: float = 1.0, c: float = 0.5):
        super().__init__()
        self.a, self.b, self.c = a, b, c
        self.bce = nn.BCELoss()
        self.dice = SoftDiceLoss()
        self.topo = SoftclDiceLoss()
    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor, y_skel: torch.Tensor):
        with torch.amp.autocast("cuda", enabled=False):
            y_safe = torch.clamp(y_pred.float(), 1e-6, 1.0 - 1e-6)
            l_bce = self.bce(y_safe, y_true.float())
        l_dice = self.dice(y_pred, y_true)
        l_topo = self.topo(y_pred, y_true, y_skel)
        return (self.a * l_bce) + (self.b * l_dice) + (self.c * l_topo)

criterion = SafeCompositeTopologicalSegmentationLoss(a=1.0, b=1.0, c=0.5)
optimizer = torch.optim.AdamW(segmenter.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60, eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

def compute_metrics(pred_bin: np.ndarray, true_bin: np.ndarray):
    inter = np.logical_and(pred_bin, true_bin).sum()
    union = np.logical_or(pred_bin, true_bin).sum()
    iou = 1.0 if union == 0 else inter / union
    f1 = 1.0 if (pred_bin.sum() + true_bin.sum()) == 0 else (2.0 * inter) / (pred_bin.sum() + true_bin.sum())
    return iou, f1

## 9. Chronological Stage 2 Training Loop (60 Epochs)

In [ ]:
import time
from tqdm.notebook import tqdm

NUM_EPOCHS = 60
best_val_iou = 0.0
history = {"train_loss": [], "val_iou": [], "val_f1": []}

print(f"Starting Stage 2 Dual-Stream LTCA Training for {NUM_EPOCHS} epochs...")
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    segmenter.train()
    epoch_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch [{epoch:02d}/{NUM_EPOCHS}]", leave=False)
    for rgb, mask, skel in pbar:
        rgb = rgb.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        skel = skel.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            pred_map, alpha, _ = segmenter(rgb)
            if pred_map.shape[-2:] != mask.shape[-2:]:
                pred_map = F.interpolate(pred_map, size=mask.shape[-2:], mode="bilinear", align_corners=False)
            loss = criterion(pred_map, mask, skel)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    history["train_loss"].append(float(avg_loss))

    if epoch % 5 == 0 or epoch == NUM_EPOCHS:
        segmenter.eval()
        total_iou, total_f1, count = 0.0, 0.0, 0
        with torch.no_grad():
            for rgb, mask, _ in val_loader:
                rgb, mask = rgb.to(device), mask.to(device)
                with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                    pred, _, _ = segmenter(rgb)
                    if pred.shape[-2:] != mask.shape[-2:]:
                        pred = F.interpolate(pred, size=mask.shape[-2:], mode="bilinear", align_corners=False)

                pred_np = (pred.cpu().numpy() >= 0.5).squeeze(1)
                mask_np = (mask.cpu().numpy() > 0.5).squeeze(1)
                for i in range(rgb.size(0)):
                    iou, f1 = compute_metrics(pred_np[i], mask_np[i])
                    total_iou += iou
                    total_f1 += f1
                count += rgb.size(0)

        val_iou = float((total_iou / count) * 100.0)
        val_f1 = float((total_f1 / count) * 100.0)
        history["val_iou"].append(val_iou)
        history["val_f1"].append(val_f1)

        print(f"--> [Epoch {epoch:02d}/{NUM_EPOCHS}] Train Loss: {avg_loss:.4f} | Val mIoU: {val_iou:.2f}% | Val F1: {val_f1:.2f}%")

        if val_iou > best_val_iou:
            best_val_iou = val_iou
            torch.save({
                "epoch": epoch,
                "model_state_dict": segmenter.state_dict(),
                "val_iou": float(val_iou),
                "val_f1": float(val_f1),
            }, WEIGHTS_DIR / "c2_ltca_segmenter_best.pth")
            print(f"    * Best C2 segmenter model saved! (mIoU: {val_iou:.2f}%)")

total_m = (time.time() - start_time) / 60.0
print(f"Stage 2 training completed in {total_m:.1f} mins. Peak Val mIoU: {best_val_iou:.2f}%")

## 10. Test Set Evaluation & Tri-Pillar Benchmark Report

In [ ]:
best_ckpt = torch.load(WEIGHTS_DIR / "c2_ltca_segmenter_best.pth", map_location=device, weights_only=False)
segmenter.load_state_dict(best_ckpt["model_state_dict"])
segmenter.eval()

total_iou, total_f1, total_topo, count = 0.0, 0.0, 0.0, 0
with torch.no_grad():
    for rgb, mask, skel in test_loader:
        rgb, mask, skel = rgb.to(device), mask.to(device), skel.to(device)
        pred, _, _ = segmenter(rgb)
        if pred.shape[-2:] != mask.shape[-2:]:
            pred = F.interpolate(pred, size=mask.shape[-2:], mode="bilinear", align_corners=False)

        pred_np = (pred.cpu().numpy() >= 0.5).squeeze(1)
        mask_np = (mask.cpu().numpy() > 0.5).squeeze(1)
        skel_np = (skel.cpu().numpy() > 0.5).squeeze(1)

        for i in range(rgb.size(0)):
            iou, f1 = compute_metrics(pred_np[i], mask_np[i])
            total_iou += iou
            total_f1 += f1

            pred_skel = morphological_thinning((pred_np[i] * 255).astype(np.uint8))
            inter_skel = np.logical_and(pred_skel > 0, mask_np[i]).sum()
            total_skel_pts = (pred_skel > 0).sum()
            topo_cont = 1.0 if total_skel_pts == 0 else inter_skel / total_skel_pts
            total_topo += topo_cont
        count += rgb.size(0)

test_iou = (total_iou / count) * 100.0
test_f1 = (total_f1 / count) * 100.0
test_topo = (total_topo / count) * 100.0

# VRAM & Latency Benchmark
torch.cuda.reset_peak_memory_stats(device)
dummy_input = torch.randn(1, 3, 256, 256, device=device)
for _ in range(10):
    _ = segmenter(dummy_input)
torch.cuda.synchronize()

t_start = time.time()
iterations = 100
for _ in range(iterations):
    _ = segmenter(dummy_input)
torch.cuda.synchronize()
latency_ms = ((time.time() - t_start) / iterations) * 1000.0
peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024**2)

print("=" * 65)
print("   ROADLENS COMPONENT C2: MULTI-PILLAR VALIDATION REPORT")
print("=" * 65)
print(f"Pillar 1: Spatial Boundary Overlap")
print(f"  - Mean IoU (mIoU):               {test_iou:.2f}%")
print(f"  - F1-Score / Dice:               {test_f1:.2f}%")
print(f"Pillar 2: Structural Line Continuity")
print(f"  - Topological Continuity:        {test_topo:.2f}%")
print(f"Pillar 3: Edge Hardware Efficiency")
print(f"  - Peak VRAM Footprint:           {peak_vram_mb:.1f} MB")
print(f"  - Inference Latency / Crop:      {latency_ms:.2f} ms")
print("=" * 65)

## 11. Demonstrating the 4-Part Output Suite

In [ ]:
rgb_t, mask_t, _ = test_dataset[0]
rgb_batch = rgb_t.unsqueeze(0).to(device)

with torch.no_grad():
    prob_map, alpha, pseudo_thm = segmenter(rgb_batch)
    if prob_map.shape[-2:] != rgb_t.shape[-2:]:
        prob_map = F.interpolate(prob_map, size=rgb_t.shape[-2:], mode="bilinear", align_corners=False)

rgb_disp = (denorm(rgb_t).transpose(1, 2, 0) * 255).astype(np.uint8)
prob_np = prob_map.squeeze().cpu().numpy()

binary_mask = (prob_np >= 0.5).astype(np.uint8) * 255
skeleton_1px = (morphological_thinning(binary_mask) > 0).astype(np.uint8) * 255

np.savez_compressed(SUITE_DEMOS_DIR / "sample_confidence.npz", confidence=prob_np)

red_overlay = rgb_disp.copy()
mask_bool = binary_mask > 0
red_overlay[mask_bool] = (0.60 * rgb_disp[mask_bool] + 0.40 * np.array([255, 0, 0])).astype(np.uint8)
contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(red_overlay, contours, -1, (255, 30, 30), thickness=1)
skel_bool = skeleton_1px > 0
red_overlay[skel_bool] = [255, 255, 0]

cv2.imwrite(str(SUITE_DEMOS_DIR / "output_a_binary_mask.png"), binary_mask)
cv2.imwrite(str(SUITE_DEMOS_DIR / "output_b_skeleton_1px.png"), skeleton_1px)
cv2.imwrite(str(SUITE_DEMOS_DIR / "output_d_red_overlay.png"), cv2.cvtColor(red_overlay, cv2.COLOR_RGB2BGR))

fig, ax = plt.subplots(1, 4, figsize=(18, 5))
ax[0].imshow(binary_mask, cmap="gray")
ax[0].set_title("Output A: 2D Binary Mask\n(For C3 Depth/Volume)")
ax[0].axis("off")

ax[1].imshow(skeleton_1px, cmap="hot")
ax[1].set_title("Output B: 1-Pixel Skeleton\n(For C3 GIS Crack/Pothole Outline)")
ax[1].axis("off")

ax[2].imshow(prob_np, cmap="viridis")
ax[2].set_title("Output C: Soft Confidence Map\n(Continuous Probability [.npz])")
ax[2].axis("off")

ax[3].imshow(red_overlay)
ax[3].set_title("Output D: Red Visual Overlay\n(For C4 / Municipal Dashboard)")
ax[3].axis("off")

plt.tight_layout()
plt.savefig(SUITE_DEMOS_DIR / "output_suite_visualization.png", dpi=200)
plt.show()

print(f"\nAll 4-Part Precision Output Suite artifacts saved to:\n{SUITE_DEMOS_DIR}")